# Практика: fit, predict_proba и выбор порога

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_bank_csv() -> Path:
    for path in (Path("bank_marketing_slim.csv"), Path("../../data/bank_marketing_slim.csv")):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"


CSV_PATH = find_bank_csv()
df = pd.read_csv(CSV_PATH)
target = df["y"].eq("yes").astype(int)
assert len(df) > 0 and set(target.unique()) == {0, 1}
assert "duration" in df.columns  # колонка видна только для разбора утечки
print(f"Строк: {len(df)}; доля yes: {target.mean():.3f}")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


## 1. Явный список безопасных признаков

In [ ]:
FEATURE_COLUMNS = None  # TODO
assert isinstance(FEATURE_COLUMNS, list) and len(FEATURE_COLUMNS) >= 5
assert "duration" not in FEATURE_COLUMNS and "y" not in FEATURE_COLUMNS
assert set(FEATURE_COLUMNS) <= set(df.columns)


## 2. Кодирование категорий

In [ ]:
X = None  # TODO: pd.get_dummies(..., drop_first=True)
assert isinstance(X, pd.DataFrame) and len(X) == len(df)
assert "duration" not in X.columns
assert X.select_dtypes(exclude="number").shape[1] == 0
assert not X.isna().any().any()


## 3. Стратифицированный train/test

In [ ]:
X_train, X_test, y_train, y_test = None, None, None, None  # TODO
assert X_train is not None and len(X_train) + len(X_test) == len(X)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert abs(float(y_train.mean()) - float(y_test.mean())) < 0.05


## 4. Обучение модели

In [ ]:
model = None  # TODO: LogisticRegression(max_iter=2000)
assert model is not None and hasattr(model, "coef_")
assert model.n_features_in_ == X_train.shape[1]


## 5. Вероятность положительного класса

In [ ]:
proba_test = None  # TODO: второй столбец predict_proba
assert isinstance(proba_test, np.ndarray) and len(proba_test) == len(y_test)
assert np.all((0 <= proba_test) & (proba_test <= 1))
assert np.std(proba_test) > 0


## 6. Функция метрик для порога

In [ ]:
def score_at_threshold(y_true, proba, threshold):
    # TODO: dict threshold, selected, precision, recall, f1
    ...


row_05 = score_at_threshold(y_test, proba_test, 0.5)
assert set(row_05) == {"threshold", "selected", "precision", "recall", "f1"}
assert 0 <= row_05["f1"] <= 1


## 7. Таблица порогов

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_table = None  # TODO: DataFrame из score_at_threshold
assert isinstance(threshold_table, pd.DataFrame) and len(threshold_table) == len(thresholds)
assert threshold_table["selected"].is_monotonic_decreasing
assert threshold_table[["precision", "recall", "f1"]].apply(lambda s: s.between(0, 1).all()).all()


## 8. Порог при ограничении recall

Выберите среди строк с recall ≥ 0.70 строку с наибольшей precision.

In [ ]:
eligible = None  # TODO
chosen_row = None  # TODO: Series
assert isinstance(eligible, pd.DataFrame) and len(eligible) > 0
assert float(chosen_row["recall"]) >= 0.70
assert float(chosen_row["precision"]) == float(eligible["precision"].max())


## 9. Самостоятельно: рекомендация кампании

In [ ]:
THRESHOLD_NOTE = ""  # TODO: числа chosen_row, компромисс и ограничение test
assert len(THRESHOLD_NOTE) >= 220
assert "recall" in THRESHOLD_NOTE.lower() and "precision" in THRESHOLD_NOTE.lower()
assert str(round(float(chosen_row["threshold"]), 2)) in THRESHOLD_NOTE
